# Grids: Using the Web UI and the API
This tutorial explains how to:
1. Use the **web UI** to:
   - list existing grids
   - upload a new grid JSON
   - inspect a grid (graph + tables)
   - delete a grid
2. Do the **same operations via the API** from Python, for automation / scripting.

## 1. Using the Web UI

![Menu](images/menu.png)

### 1.1. Navigating to the Grids page
1. Open your browser and go to:
   ```text
   http://localhost:8000/ (X)
2. On the home page you will see a card titled “Grids”.
Click that card

You should now be on the Grids page.

![Grids Menu](images/grids.png)

### 1.2. Understanding the Grids list (`/grid/ui`)
On `/grid/ui` you will see:
- A **table** with two columns:
  - **Grid ID** – the identifier of each grid stored in the database.
  - **Actions** – currently a **Delete** button.
- A **card** titled **“Add”**, which is the upload area for new grids.

Behaviour:
- If no grids exist yet, the table shows:  
  `No grids found`
- Each grid ID is a **clickable link** that takes you to that grid’s detail page.
- The red **Delete** button calls the backend `DELETE /grid/{grid_id}` and asks for confirmation.

This UI is powered by:
- `GET /grid/ui` → renders the page
- `POST /grid/` → upload JSON
- `DELETE /grid/{grid_id}` → delete grid

### 1.3. Uploading a new grid via UI

Still on `/grid/ui`, look at the **“Add”** section:

1. Prepare a `.json` file that defines your grid - it must conform to the `Grid` schema, which includes:

- grid_id,
- nodes, 
- connections, 
- cables.

Example:
```json
{
  "grid_id": "grid_001",
  "nodes": [
    {
      "node_id": "PT",
      "coord_lat": 38.7223,
      "coord_lon": -9.1393,
      "coord_error": 5
    },
    {
      "node_id": "NODE000000002",
      "coord_lat": 38.7260,
      "coord_lon": -9.1400,
      "coord_error": null
    },
    {
      "node_id": "NODE000000003",
      "coord_lat": null,
      "coord_lon": null,
      "coord_error": null
    }
  ],
  "connections": [
    {
      "connection_id": "LineUID000001",
      "from_node_id": "PT",
      "to_node_id": "NODE000000002",
      "length": 120.5,
      "cable_id": "TypeLine00001"
    },
    {
      "connection_id": "LineUID000002",
      "from_node_id": "NODE000000002",
      "to_node_id": "NODE000000003",
      "length": 85.0,
      "cable_id": "TypeLine00002"
    }
  ],
  "cables": [
    {
      "cable_id": "TypeLine00001",
      "r_imp_real": 0.123,
      "r_imp_imag": 0.045,
      "s_imp_real": 0.120,
      "s_imp_imag": 0.044,
      "t_imp_real": 0.119,
      "t_imp_imag": 0.043,
      "r_nom_curr": 150.0,
      "s_nom_curr": 150.0,
      "t_nom_curr": 150.0
    },
    {
      "cable_id": "TypeLine00002",
      "r_imp_real": 0.200,
      "r_imp_imag": 0.060,
      "s_imp_real": 0.198,
      "s_imp_imag": 0.059,
      "t_imp_real": 0.197,
      "t_imp_imag": 0.058,
      "r_nom_curr": 100.0,
      "s_nom_curr": 100.0,
      "t_nom_curr": 100.0
    }
  ]
}


2. In the **blue dashed box** (“Drop the JSON file here…”):
   - Either **drag and drop** your `.json` file, **or**
   - Click the box to open a file picker and choose your `.json`.
3. Choose the intended usage for each problem area:
   - **Phase detection**
   - **Topology detection**
   - **Voltage control**
   - **State estimation**
   
   For each one, the UI lets you choose one of:
   - `NONE`
   - `TEST`
   - `TRAIN`
   - `TEST_TRAIN`
   
   Internally, each selector is converted into a pair of query parameters:
   - `TEST` → `*_test=YES`, `*_train=NO`
   - `TRAIN` → `*_test=NO`, `*_train=YES`
   - `TEST_TRAIN` → `*_test=YES`, `*_train=YES`
   - `NONE` → `*_test=NO`, `*_train=NO`
4. After selecting a file:
   - The label changes from `No file selected` to the file name.
   - The **Upload** button becomes enabled.
5. Click **Upload**.

![Uploaded grid](images/uploaded_grid.png)

What happens under the hood:

- The UI creates a `FormData` object and sends it to:

  ```text
  POST /grid/
- The backend:
   - Reads the file
   - Parses the JSON
   - Validates it against the Grid schema
   - Creates the necessary database structures for that grid
   - Stores the grid data
   - Stores the usage flags derived from the selected options

If everything goes well:

- A message appears under the button (something like "Grid 'my_grid_id' registered successfully.")
- The page reloads and your new grid_id appears in the table.

If something fails (invalid JSON, schema error, DB error):
- The message shows the error text coming from the API.

### 1.4. Inspecting a grid (graph and tables)
From `/grid/ui`:
1. Click on a **Grid ID** in the table, for example `my_grid_id`.
This opens:
```text
http://localhost:8000/grid/ui/my_grid_id

On this grid detail page you will see:

- A header with:
    - ← Back to Grids link (returns to /grid/ui)
    - API Docs link (goes to /docs)
- A “Graph preview” card:
    - A small toolbar with:
        - layout: tree | spring | line
        (links change the layout of the grid)
    - A graph image:
        - ![Grid Graph](images/graph_preview.png)
        - This image is generated server-side with NetworkX and Matplotlib from the "Connection" table.
- One or more data table cards:
    - Currently displaying "Node" and "Connection" tables.
    ![Nodes](images/nodes.png)
    ![Connections](images/connections.png)
    - For each table:
        - Title: e.g. Node (X) rows
        - A HTML table with columns.

Use this page to quickly check:
- How many nodes and connections the grid has.
- Basic topology in the graph preview.
- Whether your uploaded JSON looks as expected.

### 1.5. Deleting a grid from the UI
Back on `/grid/ui`:
1. Find the row of the grid you want to remove.
2. Click the red **Delete** button.
3. A confirmation dialog appears:
   > Delete grid "my_grid_id" and all associated data?
   ![DELETE](images/delete.png)
4. Click **OK** to proceed.
If the deletion succeeds, you will see an alert and the page reloads; the grid disappears from the list.
Internally, this calls:
- `DELETE /grid/{grid_id}`
Which removes all grid-related data.

## 2. Doing the same from Python (API usage)
The UI is great for interactive use.  
For automation, experiments or integration with other tools, you can use the API directly.
The endpoints:
- `POST /grid/` – register a new grid from JSON file
- `GET /grid/data/{grid_id}?table=Node|Cable|Connection|grids` – read data
- `DELETE /grid/{grid_id}` – delete grid
- `GET /grid/ui/{grid_id}/graph.png` – fetch PNG graph (optional)

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
BASE_URL = "http://localhost:8000"  # adapt if needed
def check_response(resp: requests.Response):
    """Helper to raise nice errors and return parsed JSON or text."""
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        try:
            print("Error payload:", resp.json())
        except Exception:
            print("Raw response:", resp.text)
        raise e
    try:
        return resp.json()
    except Exception:
        return resp.text

### 2.1. Register a grid via API (`POST /grid/`)

This endpoint registers a grid from a JSON file and stores its usage metadata for the supported problem areas.

The request includes:
- one uploaded JSON file (`file`)
- eight query parameters describing whether the grid is used for **test** and/or **train** in each area:

    - `phase_detection_test`
    - `phase_detection_train`
    - `topology_detection_test`
    - `topology_detection_train`
    - `voltage_control_test`
    - `voltage_control_train`
    - `state_estimation_test`
    - `state_estimation_train`

Each of these parameters accepts:
- `"YES"`
- `"NO"`

Example request:

```text
POST /grid/?phase_detection_test=YES&phase_detection_train=NO&topology_detection_test=NO&topology_detection_train=YES&voltage_control_test=NO&voltage_control_train=NO&state_estimation_test=YES&state_estimation_train=YES

In [ ]:
def register_grid_from_file(
    json_path: str | Path,
    *,
    phase_detection_test: str = "NO",
    phase_detection_train: str = "NO",
    topology_detection_test: str = "NO",
    topology_detection_train: str = "NO",
    voltage_control_test: str = "NO",
    voltage_control_train: str = "NO",
    state_estimation_test: str = "NO",
    state_estimation_train: str = "NO",
):
    json_path = Path(json_path)
    if not json_path.exists():
        raise FileNotFoundError(json_path)

    allowed = {"YES", "NO"}
    params = {
        "phase_detection_test": phase_detection_test,
        "phase_detection_train": phase_detection_train,
        "topology_detection_test": topology_detection_test,
        "topology_detection_train": topology_detection_train,
        "voltage_control_test": voltage_control_test,
        "voltage_control_train": voltage_control_train,
        "state_estimation_test": state_estimation_test,
        "state_estimation_train": state_estimation_train,
    }

    invalid = {k: v for k, v in params.items() if v not in allowed}
    if invalid:
        raise ValueError(f"All usage flags must be 'YES' or 'NO'. Invalid values: {invalid}")

    url = f"{BASE_URL}/grid/"
    with open(json_path, "rb") as f:
        files = {"file": (json_path.name, f, "application/json")}
        resp = requests.post(url, params=params, files=files)
    return check_response(resp)

# Example (uncomment and adapt path):
# register_grid_from_file(
#     "data/my_grid.json",
#     phase_detection_test="YES",
#     phase_detection_train="NO",
#     topology_detection_test="NO",
#     topology_detection_train="YES",
#     voltage_control_test="NO",
#     voltage_control_train="NO",
#     state_estimation_test="YES",
#     state_estimation_train="YES",
# )

### 2.2. Read grid tables (`GET /grid/data/{grid_id}`)
You can fetch the tables that the UI is showing.
Valid table names:
- `"Node"`
- `"Connection"`
- `"Cable"`
- `"grids"`

In [ ]:
def get_grid_table(grid_id: str, table: str) -> pd.DataFrame:
    if table not in {"Node", "Cable", "Connection", "grids"}:
        raise ValueError("table must be one of: 'Node', 'Cable', 'Connection', 'grids'")
    url = f"{BASE_URL}/grid/data/{grid_id}"
    params = {"table": table}
    resp = requests.get(url, params=params)
    data = check_response(resp)
    records = data.get("records", [])
    return pd.DataFrame(records)
# Example (after registering a grid):
# grid_id = "my_grid_id"
# node_df = get_grid_table(grid_id, "Node")
# node_df.head()

### 2.3. Delete a grid via API (`DELETE /grid/{grid_id}`)

In [ ]:
def delete_grid(grid_id: str):
    url = f"{BASE_URL}/grid/{grid_id}"
    resp = requests.delete(url)
    return check_response(resp)

# Example (destructive!):
# delete_grid("my_grid_id")

### 2.4. Fetch the graph image in the notebook
This uses the same endpoint the UI uses to show the graph.

In [ ]:
def show_grid_graph_png(grid_id: str):
    url = f"{BASE_URL}/grid/ui/{grid_id}/graph.png"
    resp = requests.get(url)
    resp.raise_for_status()
    display(Image(resp.content))
# Example:
# show_grid_graph_png("my_grid_id")

## 3. Workflow summary
### In the browser (UI):
1. Go to `http://localhost:8000/grid/ui`
2. Upload grid JSON in the **Add** card.
3. Select the usage of that grid for:
   - Phase detection
   - Topology detection
   - Voltage control
   - State estimation
4. Click your **Grid ID** to inspect:
   - Graph preview
   - Node / Connection tables
5. Delete the grid with the **Delete** button if needed.
### In Python (API):
```python
register_grid_from_file(
    "data/my_grid.json",
    phase_detection_test="YES",
    phase_detection_train="NO",
    topology_detection_test="NO",
    topology_detection_train="YES",
    voltage_control_test="NO",
    voltage_control_train="NO",
    state_estimation_test="YES",
    state_estimation_train="YES",
)

node_df = get_grid_table("my_grid_id", "Node")
show_grid_graph_png("my_grid_id")
delete_grid("my_grid_id")

---